# Run Benchmarks


In [ ]:

BASE_DIR = Path.cwd()
BENCHMARK_DIR = BASE_DIR / 'Benchmarks'
OUTPUT_DIR = BASE_DIR / 'Outputs' / 'Raw Results'
CONFIG_PATH = BASE_DIR / 'config.json'


PROVIDER = 'provider-placeholder'
KEY_NAMES = None  # Example: ['key_1', 'key_2']; None means all available keys.
PROVIDER_URL_FIELDS = {
    'provider-placeholder': 'provider-placeholder_URL',
}
PROVIDER_API_MODES = {
    'provider-placeholder': 'chat_completions',

}

API_MODE = PROVIDER_API_MODES[PROVIDER]


LLM_MODEL = 'model-placeholder'
TEMPERATURE = 0.3
MAX_TOKENS = None
MAX_RETRIES = 5
RETRY_SLEEP_SECONDS = 5
workers = 10  # Total concurrency = number of API keys * workers; 5 keys and workers=2 means 10 concurrent requests.
save_every = 20


MAX_CASES_PER_BENCHMARK = None  # Set to an integer for a smoke test.
OVERWRITE_SUCCESS = False
RAISE_ON_ROW_FAIL = False
VALIDATE_JSON_OUTPUT = True


SAFE_MODEL_NAME = re.sub(r'[^0-9A-Za-z._-]+', '_', LLM_MODEL).strip('_')
OUTPUT_WORKBOOK_PATH = OUTPUT_DIR / f'{SAFE_MODEL_NAME}_benchmark_outputs.xlsx'
OUTPUT_COLUMN = f'{SAFE_MODEL_NAME}_output'
STATUS_COLUMN = f'{SAFE_MODEL_NAME}_status'
ERROR_COLUMN = f'{SAFE_MODEL_NAME}_error'
UPDATED_AT_COLUMN = f'{SAFE_MODEL_NAME}_updated_at'
ELAPSED_SECONDS_COLUMN = f'{SAFE_MODEL_NAME}_elapsed_seconds'


BENCHMARKS = [
    {
        'name': 'MTbench',
        'source_path': BENCHMARK_DIR / 'MTbench.xlsx',
        'source_sheet_name': 'cases',
        'sheet_name': 'MTbench_cases',
        'id_column': 'Case_ID',
        'prompt_template': MTBENCH_PROMPT,
        'expects_json': True,
    },
    {
        'name': 'Extend Treatment Goal Matching',
        'source_path': BENCHMARK_DIR / 'Extend Treatment Goal Matching.xlsx',
        'source_sheet_name': 'TGM_cases',
        'sheet_name': 'TGM_cases',
        'id_column': 'TGM_ID',
        'prompt_template': EXTEND_MTBENCH_PROMPT,
        'expects_json': True,
    },
    {
        'name': 'Extend Song Matching',
        'source_path': BENCHMARK_DIR / 'Extend Song Matching.xlsx',
        'source_sheet_name': 'SM_cases',
        'sheet_name': 'SM_cases',
        'id_column': 'SM_ID',
        'prompt_template': EXTEND_MTBENCH_PROMPT,
        'expects_json': True,
    },
    {
        'name': 'Extend Prescription Completion',
        'source_path': BENCHMARK_DIR / 'Extend Prescription Completion.xlsx',
        'source_sheet_name': 'PC_cases',
        'sheet_name': 'PC_cases',
        'id_column': 'PC_ID',
        'prompt_template': EXTEND_MTBENCH_PROMPT,
        'expects_json': True,
    },
    {
        'name': 'Extend Clinical Context Matching',
        'source_path': BENCHMARK_DIR / 'Extend Clinical Context Matching.xlsx',
        'source_sheet_name': 'CCM_cases',
        'sheet_name': 'CCM_cases',
        'id_column': 'CCM_ID',
        'prompt_template': EXTEND_MTBENCH_PROMPT,
        'expects_json': True,
    },
]


In [ ]:

def get_header_map(ws):
    return {str(cell.value).strip(): cell.column for cell in ws[1] if cell.value is not None}


def ensure_column(ws, header_name):
    header_map = get_header_map(ws)
    if header_name in header_map:
        return header_map[header_name]
    new_col = ws.max_column + 1
    ws.cell(row=1, column=new_col, value=header_name)
    return new_col


def legacy_output_path_for(source_path):
    return OUTPUT_DIR / f'{source_path.stem}__{SAFE_MODEL_NAME}.xlsx'



def source_sheet_name_for(cfg):
    return cfg.get('source_sheet_name', cfg['sheet_name'])


def copy_sheet_to_workbook(target_wb, source_path, source_sheet_name, target_sheet_name):
    source_wb = load_workbook(source_path)
    if source_sheet_name not in source_wb.sheetnames:
        raise ValueError(f'Sheet not found in {source_path}: {source_sheet_name}')

    source_ws = source_wb[source_sheet_name]
    target_ws = target_wb.create_sheet(target_sheet_name)
    for row in source_ws.iter_rows():
        for source_cell in row:
            target_cell = target_ws.cell(row=source_cell.row, column=source_cell.column, value=source_cell.value)
            if source_cell.has_style:
                target_cell.font = copy(source_cell.font)
                target_cell.fill = copy(source_cell.fill)
                target_cell.border = copy(source_cell.border)
                target_cell.alignment = copy(source_cell.alignment)
                target_cell.protection = copy(source_cell.protection)
                target_cell.number_format = source_cell.number_format
            if source_cell.comment:
                target_cell.comment = copy(source_cell.comment)
            if source_cell.hyperlink:
                target_cell._hyperlink = copy(source_cell.hyperlink)

    for key, dimension in source_ws.column_dimensions.items():
        target_ws.column_dimensions[key].width = dimension.width
    for key, dimension in source_ws.row_dimensions.items():
        target_ws.row_dimensions[key].height = dimension.height
    for merged_range in source_ws.merged_cells.ranges:
        target_ws.merge_cells(str(merged_range))
    target_ws.freeze_panes = source_ws.freeze_panes
    if source_ws.auto_filter.ref:
        target_ws.auto_filter.ref = source_ws.auto_filter.ref



def prepare_output_workbook(benchmarks):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_WORKBOOK_PATH
    if output_path.exists():
        wb = load_workbook(output_path)
        print(f'Resuming existing output workbook: {output_path}')
    else:
        wb = Workbook()
        default_sheet = wb.active
        wb.remove(default_sheet)
        print(f'Created output workbook: {output_path}')

    for cfg in benchmarks:
        target_sheet_name = cfg['sheet_name']
        if target_sheet_name in wb.sheetnames:
            continue

        source_sheet_name = source_sheet_name_for(cfg)
        legacy_path = legacy_output_path_for(cfg['source_path'])
        copy_from_path = legacy_path if legacy_path.exists() else cfg['source_path']
        copy_sheet_to_workbook(wb, copy_from_path, source_sheet_name, target_sheet_name)
        print(f'Added sheet {target_sheet_name} from {copy_from_path.name}')

    wb.save(output_path)
    return output_path



def load_case_rows(ws, id_column, status_col):
    header_map = get_header_map(ws)
    required = ['Question']
    if id_column:
        required.append(id_column)
    missing = [name for name in required if name not in header_map]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    question_col = header_map['Question']
    id_col = header_map.get(id_column) if id_column else None

    rows = []
    for row_idx in range(2, ws.max_row + 1):
        row_id = ws.cell(row=row_idx, column=id_col).value if id_col else row_idx
        question = ws.cell(row=row_idx, column=question_col).value
        status = ws.cell(row=row_idx, column=status_col).value
        if row_id is None and question is None:
            continue
        rows.append({
            'row_idx': row_idx,
            'row_id': '' if row_id is None else str(row_id).strip(),
            'question': '' if question is None else str(question),
            'status': '' if status is None else str(status).strip().lower(),
        })

    if MAX_CASES_PER_BENCHMARK is not None:
        rows = rows[:MAX_CASES_PER_BENCHMARK]
    return rows


def should_run(row):
    if OVERWRITE_SUCCESS:
        return True
    return row['status'] != 'success'



In [ ]:

def run_all_benchmarks():
    validate_prompt_templates(BENCHMARKS)
    output_path = prepare_output_workbook(BENCHMARKS)
    client_slots = make_client_slots()
    results = []
    for cfg in BENCHMARKS:
        results.append(run_single_benchmark(cfg, client_slots, output_path))
    return results


results = run_all_benchmarks()
results
